In [ ]:
import numpy as np

In [ ]:
# Set seed for reproducible initial weights
np.random.seed(42)

# The sequence and it's predictions :
corpus = ["a","b","c"]
targets = ["b","c"]

# These are raw values which cannot be passed so encode them into numeric value
# For extraction of such words we have a systematic approach of extracting all the unique values from the text at once to build vocabulary

vocabulary = sorted(list(set(" ".join(corpus).split())))

vocab_size = len(vocabulary)

# Create a set of dictionaries to map the tokens to unique id
word_to_idx = {word : i for i,word in enumerate(vocabulary)}
idx_to_word = {idx : word for idx,word in enumerate(vocabulary)}

# Next step is to conver our text in Numeric form for which we will just onehotencode them into dimension i .
# Note :  i is the no of input values and also the max words in corpus

# X_words will be ['a', 'b'], y_words will be ['b', 'c']
X_words = corpus[:-1]
y_words = corpus[1:]

# Convert your word sequences into integer index lists using your dictionary
X_indices = [word_to_idx[word] for word in corpus[:-1]]  # [0, 1]
y_indices = [word_to_idx[word] for word in corpus[1:]]   # [1, 2]

# The Trick: Use the indices to grab rows from a 3x3 Identity Matrix
X_train = np.eye(vocab_size)[X_indices]
y_train = np.eye(vocab_size)[y_indices]

In [ ]:
X_train

array([[1., 0., 0.],
       [0., 1., 0.]])

In [ ]:
y_train

array([[0., 1., 0.],
       [0., 0., 1.]])

In [ ]:
# Now set the initial values of each parameter -- used to iterate later.
input_dim = 3
hidden_dim = 2
output_dim = 3

# 1. Weights from Input Layer to Hidden Layer (Shape: 3 inputs x 2 hidden nodes)
W_xh = np.random.randn(input_dim, hidden_dim) * 0.01 # Initialize with a random small number
b_h = np.zeros((1, hidden_dim))

# 2. Weights from Hidden Layer to Hidden State (Shape: 2 hidden nodes x 3 outputs)
W_hh = np.random.randn(hidden_dim,hidden_dim) * 0.01  # Similar to w1 initialization
h_0 = np.zeros((hidden_dim,1)) # First default hidden state

# 3. Weights from Hidden Layer to Output Layer (Shape: 2 hidden nodes x 3 outputs)
W_ho = np.random.randn(hidden_dim, output_dim) * 0.01 # Similar to w1 initialization
b_o = np.zeros((1, output_dim))

In [ ]:
# =======================================================
# THE FORWARD PASS (First Time Step: Inputting "a")
# =======================================================

In [ ]:
# Prepare time-step 1 column vectors

t = 1

x_1 = X_train[0].reshape(-1, 1)          # Shape (3, 1) - One-hot vector for "a"
h_prev = h_0.reshape(hidden_dim, 1)      # Shape (2, 1) - Initial hidden state

# Step 1: Compute Hidden State using transposes
raw_h = np.dot(W_xh.T, x_1) + np.dot(W_hh.T, h_prev) + b_h.T
h_1 = np.tanh(raw_h)                     # Shape (2, 1)

# Step 2: Compute Unnormalized Output Scores (Logits)
logits = np.dot(W_ho.T, h_1) + b_o.T     # Shape (3, 1)

# Step 3: Compute Softmax Probabilities
exp_logits = np.exp(logits)
y_hat = exp_logits / np.sum(exp_logits)  # Shape (3, 1)

print(f"--- Final Output Probabilities (y_hat) at  t={t} ---")
for word, prob in zip(vocabulary, y_hat.flatten()):
    print(f"  Probability of '{word}': {prob * 100:.2f}%")

--- Final Output Probabilities (y_hat) at  t=1 ---
  Probability of 'a': 33.33%
  Probability of 'b': 33.33%
  Probability of 'c': 33.33%


### Step 1: Loss Function and Gradients
We will use categorical cross-entropy loss. For a single time step, the gradient of the loss with respect to the logits is simply $(y_{hat} - y_{true})$.

In [ ]:
learning_rate = 0.1

# To store gradients
dW_xh, dW_hh, dW_ho = np.zeros_like(W_xh), np.zeros_like(W_hh), np.zeros_like(W_ho)
db_h, db_o = np.zeros_like(b_h), np.zeros_like(b_o)

# We'll need to store states for the backward pass
# In a real loop, you'd store these in a list or dict
# For this demo, we have h_0, h_1, h_2 and y_hat_1, y_hat_2

# Let's re-run a simplified forward pass to keep track of variables for BPTT
h_states = {0: h_0}
y_hats = {}

# Forward pass
for t, x_idx in enumerate(X_indices):
    xt = X_train[t].reshape(-1, 1)
    raw_h = np.dot(W_xh.T, xt) + np.dot(W_hh.T, h_states[t]) + b_h.T
    h_states[t+1] = np.tanh(raw_h)
    logits = np.dot(W_ho.T, h_states[t+1]) + b_o.T
    y_hats[t+1] = np.exp(logits) / np.sum(np.exp(logits))

print("Forward pass complete for gradient calculation.")

Forward pass complete for gradient calculation.


In [ ]:
dh_next = np.zeros_like(h_0)
loss = 0

# Backward pass (BPTT)
for t in reversed(range(len(X_indices))):
    # 1. Loss (Cross Entropy)
    y_true = y_train[t].reshape(-1, 1)
    loss += -np.sum(y_true * np.log(y_hats[t+1] + 1e-8))

    # 2. Gradient of loss w.r.t logits
    d_logits = y_hats[t+1] - y_true

    # 3. Gradients for Output Layer
    dW_ho += np.dot(h_states[t+1], d_logits.T)
    db_o += d_logits.T

    # 4. Gradient through the hidden state
    # dh consists of gradient from output AND gradient from next time step
    dh = np.dot(W_ho, d_logits) + dh_next

    # 5. Backprop through tanh
    dh_raw = (1 - h_states[t+1]**2) * dh

    # 6. Gradients for Input and Hidden weights
    db_h += dh_raw.T
    dW_xh += np.dot(X_train[t].reshape(-1, 1), dh_raw.T)
    dW_hh += np.dot(h_states[t], dh_raw.T)

    # Update dh_next for the previous time step (t-1)
    dh_next = np.dot(W_hh, dh_raw)

print(f"Total Loss: {loss:.4f}")

Total Loss: 2.1971


In [ ]:
# Update Weights using SGD
W_xh -= learning_rate * dW_xh
W_hh -= learning_rate * dW_hh
W_ho -= learning_rate * dW_ho
b_h -= learning_rate * db_h
b_o -= learning_rate * db_o

print("Weights updated successfully!")

Weights updated successfully!


In [ ]:
# Prepare time-step 2 column vectors

t = 2

x_2 = X_train[1].reshape(-1, 1)
h_prev = h_1.reshape(hidden_dim, 1)

# Step 1: Compute Hidden State using transposes
raw_h = np.dot(W_xh.T, x_2) + np.dot(W_hh.T, h_prev) + b_h.T
h_2 = np.tanh(raw_h)

# Step 2: Compute Unnormalized Output Scores (Logits)
logits = np.dot(W_ho.T, h_2) + b_o.T

# Step 3: Compute Softmax Probabilities
exp_logits = np.exp(logits)
y_hat = exp_logits / np.sum(exp_logits)

print(f"--- Final Output Probabilities (y_hat) at t={t} ---")
for word, prob in zip(vocabulary, y_hat.flatten()):
    print(f"  Probability of '{word}': {prob * 100:.2f}%")

--- Final Output Probabilities (y_hat) at t=2 ---
  Probability of 'a': 31.15%
  Probability of 'b': 34.42%
  Probability of 'c': 34.43%


In [ ]:
# Now we clearly see that the probability has clearly changed because of our weights being updated using gradients.